# Clinical Recruitment: SFT + GRPO Training
**Step 1**: Generate expert trajectories from heuristic policy  
**Step 2**: SFT the model on expert traces (learn tool format)  
**Step 3**: GRPO to refine (improve beyond heuristic)  
Requires T4+ GPU.

In [ ]:
import subprocess, sys, os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
def pip(*a): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])
pip("--upgrade", "pip")
pip("unsloth")
pip("--no-deps", "trl>=0.19.0")
pip("transformers>=5.2.0,<=5.5.0")
pip("datasets>=2.21.0", "accelerate>=0.34.0", "openenv-core[core]>=0.2.1")

In [ ]:
import json, pathlib, torch, warnings, random
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer, SFTConfig, SFTTrainer
from datasets import Dataset
from openenv.core import GenericEnvClient

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="Both `max_new_tokens`")
warnings.filterwarnings("ignore", message=".*max_length.*max_new_tokens.*")

assert torch.cuda.is_available(), "No GPU!"
gpu = torch.cuda.get_device_name(0)
print(f"GPU: {gpu} | CUDA: {torch.version.cuda} | PyTorch: {torch.__version__}")

In [ ]:
ENV_URL = os.getenv("ENV_URL", "https://pratimassaravanan-clinical-recruitment.hf.space")
MODEL_NAME = os.getenv("MODEL_NAME", "unsloth/Qwen3-4B-unsloth-bnb-4bit")
TASKS = ["easy_bench", "medium_bench", "hard_bench"]
MAX_SEQ = 2048
LORA_R, LORA_ALPHA = 16, 16
OUTPUT_DIR = "sft_grpo_clinical"
RESULTS_DIR = pathlib.Path("data/training_outputs")
SYSTEM_PROMPT = ("You are a long-horizon clinical trial recruitment agent. "
                 "Use tools to manage screening, recontact, allocation, planning, and memory.")
_H, _C = "noise_dominant", 0.6
print(f"Model: {MODEL_NAME}")

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=MAX_SEQ, load_in_4bit=True, dtype=None)
model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    bias="none", use_gradient_checkpointing="unsloth", random_state=3407)
model.print_trainable_parameters()

## Step 1: Generate Expert Trajectories
Run the heuristic policy against all 3 tasks, capture conversation traces for SFT.

In [ ]:
import httpx

def heuristic_action(obs, step_num, world_type_hint="noise", randomize=False):
    """Strong heuristic policy that knows the environment well.
    When randomize=True, 15% of actions are replaced with a random valid action."""
    available = obs.get("available_patients", [])
    recontact = obs.get("recontact_candidates", [])
    allocation = obs.get("allocation_candidates", [])
    sites = obs.get("site_performance", {})
    funnel = obs.get("current_funnel", {})
    constraints = obs.get("active_constraints", {})

    hyp_map = {"noise": "noise_dominant", "site_bias": "site_bias", "dropout": "dropout_dominant"}
    hyp = hyp_map.get(world_type_hint, "noise_dominant")

    # 15% random action injection when randomize is True
    if randomize and random.random() < 0.15:
        random_actions = []
        if available:
            p = random.choice(available)
            random_actions.append({"action_type": "screen_patient", "patient_id": p["id"], "hypothesis": hyp, "confidence": 0.7})
        if recontact:
            p = random.choice(recontact)
            random_actions.append({"action_type": "recontact", "patient_id": p["id"], "hypothesis": hyp, "confidence": 0.75})
        if allocation and sites:
            p = random.choice(allocation)
            s = random.choice(list(sites.keys()))
            random_actions.append({"action_type": "allocate_to_site", "patient_id": p["id"], "site_id": s, "hypothesis": hyp, "confidence": 0.8})
        random_actions.append({"action_type": "adjust_strategy", "strategy_change": random.choice(["increase_outreach", "relax_criteria", "tighten_criteria"]), "hypothesis": hyp, "confidence": 0.65})
        random_actions.append({"action_type": "plan_next_phase", "target_phase": random.choice(["screening", "conversion", "allocation"]), "plan_summary": "Randomized exploration action"})
        random_actions.append({"action_type": "summarize_and_index", "memory_key": f"rand_step_{step_num}", "memory_payload": f"Random exploration at step {step_num}"})
        random_actions.append({"action_type": "retrieve_relevant_history", "memory_query": "recent enrollment and screening progress"})
        return random.choice(random_actions)

    # Periodic memory retrieval every 20 steps
    if step_num % 20 == 0 and step_num > 0:
        return {"action_type": "retrieve_relevant_history", "memory_query": f"enrollment progress and bottlenecks at step {step_num}"}

    # Priority: allocate consented > recontact eligible > screen new > adjust strategy
    if allocation and sites:
        patient = allocation[0]
        # Pick best site
        best_site = max(sites.keys(), key=lambda s: sites[s].get("conversion_rate", 0) * sites[s].get("capacity_remaining", 0))
        return {"action_type": "allocate_to_site", "patient_id": patient["id"], "site_id": best_site, "hypothesis": hyp, "confidence": 0.8}

    if recontact:
        return {"action_type": "recontact", "patient_id": recontact[0]["id"], "hypothesis": hyp, "confidence": 0.75}

    if available and not constraints.get("regulatory_hold_days", 0):
        # Screen highest priority
        best = max(available, key=lambda p: p.get("eligibility_score", 0) * (1 - p.get("dropout_risk", 0)))
        return {"action_type": "screen_patient", "patient_id": best["id"], "hypothesis": hyp, "confidence": 0.7}

    # Memory/planning actions periodically
    if step_num % 15 == 5:
        return {"action_type": "plan_next_phase", "target_phase": "screening", "plan_summary": "Focus on screening high-priority patients"}
    if step_num % 15 == 10:
        return {"action_type": "summarize_and_index", "memory_key": f"step_{step_num}_snapshot", "memory_payload": f"enrolled={funnel.get('enrolled',0)} screened={funnel.get('screened',0)}"}

    # Strategy adjustments
    strategies = ["increase_outreach", "relax_criteria", "tighten_criteria"]
    return {"action_type": "adjust_strategy", "strategy_change": strategies[step_num % len(strategies)], "hypothesis": hyp, "confidence": 0.65}


def generate_expert_trajectory(task_id, max_steps=80, randomize=False):
    """Run heuristic policy and capture as conversation for SFT.
    Assistant messages contain raw JSON only for clean parsing."""
    client = httpx.Client(timeout=30)
    base = ENV_URL

    # Reset
    r = client.post(f"{base}/reset", params={"task_id": task_id})
    result = r.json()
    obs = result.get("observation", {})
    world_type = obs.get("world_type", "noise")

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    for step in range(max_steps):
        done = result.get("done", False)
        if done:
            break

        # Format observation as user message
        obs_summary = (f"step={obs.get('timestamp')} budget={obs.get('budget_remaining')} "
                      f"enrolled={obs.get('enrolled_so_far')}/{obs.get('target_enrollment')} "
                      f"available={len(obs.get('available_patients', []))} "
                      f"recontact={len(obs.get('recontact_candidates', []))} "
                      f"allocate={len(obs.get('allocation_candidates', []))} "
                      f"funnel={obs.get('current_funnel', {})}")
        messages.append({"role": "user", "content": f"State: {obs_summary}\nChoose next action."})

        # Get heuristic action
        action = heuristic_action(obs, step, world_type, randomize=randomize)

        # Format as assistant response: RAW JSON only
        messages.append({"role": "assistant", "content": json.dumps(action)})

        # Execute
        r = client.post(f"{base}/step", json=action)
        result = r.json()
        obs = result.get("observation", {})

    client.close()

    final_enrolled = obs.get("enrolled_so_far", 0)
    final_target = obs.get("target_enrollment", 100)
    return messages, final_enrolled, final_target


# Generate trajectories: 6 episodes per task (1 deterministic + 5 randomized) = 18 total
print("Generating expert trajectories...")
all_traces = []
for task in TASKS:
    # Episode 0: deterministic
    msgs, enrolled, target = generate_expert_trajectory(task, max_steps=80, randomize=False)
    all_traces.append(msgs)
    print(f"  {task} ep0 (deterministic): {len(msgs)} messages, enrolled={enrolled}/{target}")
    # Episodes 1-5: randomized with distinct seeds
    for ep in range(1, 6):
        random.seed(42 + ep * 100 + TASKS.index(task) * 10)
        msgs, enrolled, target = generate_expert_trajectory(task, max_steps=80, randomize=True)
        all_traces.append(msgs)
        print(f"  {task} ep{ep} (randomized): {len(msgs)} messages, enrolled={enrolled}/{target}")

print(f"\nGenerated {len(all_traces)} expert trajectories")

## Step 2: SFT on Expert Trajectories
Fine-tune the model to learn the tool-calling format and heuristic strategy.

In [ ]:
# Build SFT dataset from trajectories
sft_data = []
for trace in all_traces:
    # Convert to single text with chat template
    text = tokenizer.apply_chat_template(trace, tokenize=False, add_generation_prompt=False, enable_thinking=False)
    sft_data.append({"text": text})

sft_dataset = Dataset.from_list(sft_data)
print(f"SFT dataset: {len(sft_dataset)} examples")

# SFT training
FastLanguageModel.for_training(model)
sft_config = SFTConfig(
    output_dir=f"{OUTPUT_DIR}/sft",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-5,
    logging_steps=1,
    save_steps=50,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    warmup_steps=5,
    lr_scheduler_type="cosine",
    max_seq_length=MAX_SEQ,
    report_to="none",
    dataset_text_field="text",
)

sft_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=sft_dataset,
    args=sft_config,
)

print("Starting SFT training...")
sft_trainer.train()
print("SFT complete!")

## Step 2.5: Evaluate After SFT
Check if the model learned the tool-calling format.

In [ ]:
import re

def _parse_action(resp, obs):
    """Parse model output into an action dict. Tries JSON extraction first, then keyword fallback."""
    # Try JSON extraction first
    json_match = re.search(r'\{[^{}]*"action_type"\s*:\s*"[^"]+"[^{}]*\}', resp)
    if json_match:
        try:
            parsed = json.loads(json_match.group())
            if "action_type" in parsed:
                return parsed
        except json.JSONDecodeError:
            pass

    # Keyword fallback
    resp_lower = resp.lower()
    if "screen" in resp_lower and obs.get("available_patients"):
        return {"action_type": "screen_patient", "patient_id": obs["available_patients"][0]["id"], "hypothesis": _H, "confidence": _C}
    if "allocate" in resp_lower and obs.get("allocation_candidates") and obs.get("site_performance"):
        return {"action_type": "allocate_to_site", "patient_id": obs["allocation_candidates"][0]["id"], "site_id": list(obs["site_performance"])[0], "hypothesis": _H, "confidence": _C}
    if "recontact" in resp_lower and obs.get("recontact_candidates"):
        return {"action_type": "recontact", "patient_id": obs["recontact_candidates"][0]["id"], "hypothesis": _H, "confidence": _C}
    if "adjust" in resp_lower:
        return {"action_type": "adjust_strategy", "strategy_change": "increase_outreach", "hypothesis": _H, "confidence": _C}
    if "plan" in resp_lower:
        return {"action_type": "plan_next_phase", "target_phase": "screening", "plan_summary": "screen more"}
    if "retrieve" in resp_lower or "history" in resp_lower:
        return {"action_type": "retrieve_relevant_history", "memory_query": "recent progress"}
    if "summarize" in resp_lower or "memory" in resp_lower:
        return {"action_type": "summarize_and_index", "memory_key": "eval_snapshot", "memory_payload": "evaluation step"}
    if "stop" in resp_lower:
        return {"action_type": "stop_recruitment"}
    if obs.get("available_patients"):
        return {"action_type": "screen_patient", "patient_id": obs["available_patients"][0]["id"], "hypothesis": _H, "confidence": _C}
    return {"action_type": "adjust_strategy", "strategy_change": "increase_outreach", "hypothesis": _H, "confidence": _C}

def run_episode(mdl, tok, task="easy_bench", n=30):
    env_client = httpx.Client(timeout=30)
    r = env_client.post(f"{ENV_URL}/reset", params={"task_id": task})
    result = r.json()
    obs = result.get("observation", {})

    FastLanguageModel.for_inference(mdl)
    total_r, steps = 0.0, 0
    action_dist = {}
    for i in range(n):
        if result.get("done", False): break
        obs_text = (f"step={obs.get('timestamp')} budget={obs.get('budget_remaining')} "
                   f"enrolled={obs.get('enrolled_so_far')}/{obs.get('target_enrollment')} "
                   f"avail={len(obs.get('available_patients',[]))} recontact={len(obs.get('recontact_candidates',[]))} "
                   f"allocate={len(obs.get('allocation_candidates',[]))} funnel={obs.get('current_funnel',{})}")
        msgs = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"State: {obs_text}\nChoose next action."}]
        input_ids = tok.apply_chat_template(
            msgs, tokenize=True, add_generation_prompt=True, enable_thinking=False,
            max_length=MAX_SEQ, truncation=True, return_tensors="pt").to(mdl.device)
        with torch.no_grad():
            out = mdl.generate(input_ids=input_ids, max_new_tokens=512, do_sample=False,
                               pad_token_id=tok.pad_token_id or tok.eos_token_id)
        resp = tok.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)
        # Log first 3 outputs + every 10th
        if i < 3 or i % 10 == 0:
            print(f"    [{task} step {i}] model output: {resp[:200]}")
        try:
            action = _parse_action(resp, obs)
            atype = action.get("action_type", "unknown")
            action_dist[atype] = action_dist.get(atype, 0) + 1
            r = env_client.post(f"{ENV_URL}/step", json=action)
            result = r.json()
            obs = result.get("observation", {})
            total_r += result.get("reward", 0)
            steps += 1
        except Exception as e:
            print(f"    [{task} step {i}] error: {e}")
            break
    env_client.close()
    print(f"    [{task}] action distribution: {action_dist}")
    return {"task": task, "actions": steps, "total_reward": round(total_r, 4),
            "enrolled": obs.get("enrolled_so_far", 0), "target": obs.get("target_enrollment", 100),
            "action_dist": action_dist}

# Silence max_length vs max_new_tokens warning
model.generation_config.max_length = None

sep = "=" * 60
print(f"\n{sep}\nAFTER SFT\n{sep}")
after_sft = {}
for t in TASKS:
    r = run_episode(model, tokenizer, t, n=40)
    after_sft[t] = r
    print(f"  [{t}] enrolled={r.get('enrolled',0)}/{r.get('target',0)} reward={r.get('total_reward',0)}")

## Step 3: GRPO Refinement
Now that the model knows the tool format, GRPO can refine its strategy.

In [ ]:
class ClinicalRecruitmentToolEnv:
    """Tool-call environment using direct HTTP."""

    def __init__(self):
        self.client = httpx.Client(timeout=30)
        self.reward = self.initial_budget = 0.0
        self.done = False
        self.last_observation = None
        self.action_history = []
        self.enrollment_history = []
        self.budget_history = []
        self.hypothesis_history = []

    def reset(self, **kw):
        """Reset environment. Called by trainer lifecycle."""
        task = kw.get("task") or kw.get("task_id") or "easy_bench"
        r = self.client.post(f"{ENV_URL}/reset", params={"task_id": task})
        result = r.json()
        self.last_observation = result.get("observation", {})
        self.reward, self.done = 0.0, bool(result.get("done", False))
        self.action_history = []
        self.enrollment_history = []
        self.budget_history = []
        self.hypothesis_history = []
        obs = self.last_observation
        self.initial_budget = obs.get("budget_remaining", 0.0)
        self.enrollment_history.append(obs.get("enrolled_so_far", 0))
        self.budget_history.append(obs.get("budget_remaining", 0.0))
        return self._fmt()

    def close(self):
        """Close client. Called by trainer lifecycle."""
        try:
            self.client.close()
        except Exception:
            pass

    def screen_patient(self, patient_id: str, hypothesis: str = _H, confidence: float = _C) -> str:
        """Screen a candidate patient for eligibility.

        Args:
            patient_id: Patient id from available_patients.
            hypothesis: Current guess about dominant dynamics.
            confidence: Confidence in the hypothesis.
        """
        return self._step({"action_type": "screen_patient", "patient_id": patient_id, "hypothesis": hypothesis, "confidence": confidence})

    def recontact(self, patient_id: str, hypothesis: str = _H, confidence: float = _C) -> str:
        """Recontact an eligible patient for consent or enrollment.

        Args:
            patient_id: Patient id from recontact_candidates.
            hypothesis: Current guess about dominant dynamics.
            confidence: Confidence in the hypothesis.
        """
        return self._step({"action_type": "recontact", "patient_id": patient_id, "hypothesis": hypothesis, "confidence": confidence})

    def allocate_to_site(self, patient_id: str, site_id: str, hypothesis: str = _H, confidence: float = _C) -> str:
        """Allocate a consented patient to a recruitment site.

        Args:
            patient_id: Patient id from allocation_candidates.
            site_id: Site id from site_performance.
            hypothesis: Current guess about dominant dynamics.
            confidence: Confidence in the hypothesis.
        """
        return self._step({"action_type": "allocate_to_site", "patient_id": patient_id, "site_id": site_id, "hypothesis": hypothesis, "confidence": confidence})

    def adjust_strategy(self, strategy_change: str, hypothesis: str = _H, confidence: float = _C) -> str:
        """Adjust recruitment strategy such as increase_outreach or negotiate_site_A.

        Args:
            strategy_change: Strategy name like increase_outreach or tighten_criteria.
            hypothesis: Current guess about dominant dynamics.
            confidence: Confidence in the hypothesis.
        """
        return self._step({"action_type": "adjust_strategy", "strategy_change": strategy_change, "hypothesis": hypothesis, "confidence": confidence})

    def plan_next_phase(self, target_phase: str, plan_summary: str = "advance the bottleneck") -> str:
        """Create or revise the current high-level recruitment plan.

        Args:
            target_phase: One of screening, conversion, allocation, retention, recovery.
            plan_summary: Natural-language summary of the plan.
        """
        return self._step({"action_type": "plan_next_phase", "target_phase": target_phase, "plan_summary": plan_summary})

    def summarize_and_index(self, memory_key: str, memory_payload: str) -> str:
        """Write a summary into indexed episodic memory.

        Args:
            memory_key: Key for the indexed memory item.
            memory_payload: Summary text to store.
        """
        return self._step({"action_type": "summarize_and_index", "memory_key": memory_key, "memory_payload": memory_payload})

    def retrieve_relevant_history(self, memory_query: str) -> str:
        """Retrieve indexed memory entries relevant to the current bottleneck.

        Args:
            memory_query: Query string for indexed memory retrieval.
        """
        return self._step({"action_type": "retrieve_relevant_history", "memory_query": memory_query})

    def stop_recruitment(self) -> str:
        """End the current recruitment episode early."""
        return self._step({"action_type": "stop_recruitment"})

    def _step(self, action):
        if self.done:
            raise ValueError("Episode finished.")
        r = self.client.post(f"{ENV_URL}/step", json=action)
        result = r.json()
        self.last_observation = result.get("observation", {})
        self.reward = float(result.get("reward", 0))
        self.done = bool(result.get("done", False))
        self.action_history.append(action.get("action_type", ""))
        obs = self.last_observation
        self.enrollment_history.append(obs.get("enrolled_so_far", 0))
        self.budget_history.append(obs.get("budget_remaining", 0))
        if h := action.get("hypothesis"):
            self.hypothesis_history.append(h)
        return self._fmt()

    def _fmt(self):
        o = self.last_observation or {}
        return (f"step={o.get('timestamp')} budget={o.get('budget_remaining')} "
                f"enrolled={o.get('enrolled_so_far')}/{o.get('target_enrollment')} "
                f"avail={len(o.get('available_patients', []))} recontact={len(o.get('recontact_candidates', []))} "
                f"allocate={len(o.get('allocation_candidates', []))} funnel={o.get('current_funnel', {})}")


# ── Reward functions (dual-signature for TRL compatibility) ──────────
def reward_enrollment_progress(prompts=None, completions=None, environments=None, **_):
    """Fraction of target enrollment reached."""
    if environments is None:
        return [0.0] * len(prompts or completions)
    return [min(1.0, (e.last_observation or {}).get("enrolled_so_far", 0) / max(1, (e.last_observation or {}).get("target_enrollment", 100))) for e in environments]

def reward_budget_efficiency(prompts=None, completions=None, environments=None, **_):
    """Enrollment per unit budget spent."""
    if environments is None:
        return [0.0] * len(prompts or completions)
    r = []
    for e in environments:
        o = e.last_observation or {}
        ib = e.initial_budget or 1
        spent = max(0, ib - o.get("budget_remaining", 0))
        t = max(1, o.get("target_enrollment", 100))
        r.append(min(1.0, (o.get("enrolled_so_far", 0) / t) / (spent / ib)) if spent > 0 else 0.0)
    return r

def reward_screening_accuracy(prompts=None, completions=None, environments=None, **_):
    """Enrolled-to-screened ratio minus dropout penalty."""
    if environments is None:
        return [0.0] * len(prompts or completions)
    r = []
    for e in environments:
        f = (e.last_observation or {}).get("current_funnel", {})
        s = f.get("screened", 0)
        r.append(max(0, min(1, f.get("enrolled", 0) / s - 0.5 * f.get("dropped", 0) / s)) if s > 0 else 0.0)
    return r

def reward_action_diversity(prompts=None, completions=None, environments=None, **_):
    """Fraction of 8 action types used."""
    if environments is None:
        return [0.0] * len(prompts or completions)
    return [min(1, len(set(e.action_history)) / 8) if e.action_history else 0.0 for e in environments]

def reward_hypothesis_consistency(prompts=None, completions=None, environments=None, **_):
    """Penalizes switching, rewards correct world-type match."""
    if environments is None:
        return [0.0] * len(prompts or completions)
    r = []
    for e in environments:
        hs = e.hypothesis_history
        if len(hs) < 2:
            r.append(0.5)
            continue
        sw = sum(1 for i in range(1, len(hs)) if hs[i] != hs[i - 1])
        con = 1.0 if sw <= 1 else max(0.2, 1 - (sw - 1) * 0.2)
        wt = (e.last_observation or {}).get("world_type", "")
        bonus = 0.2 if {"dropout_dominant": "dropout", "noise_dominant": "noise", "site_bias": "site_bias"}.get(hs[-1], "") == wt and wt else 0
        r.append(min(1.0, con * 0.8 + bonus))
    return r

REWARD_FUNCS = [reward_enrollment_progress, reward_budget_efficiency,
                reward_screening_accuracy, reward_action_diversity,
                reward_hypothesis_consistency]

# ── GRPO training ──────────
ds = Dataset.from_dict({
    "prompt": [[{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": "Improve recruitment."}] for _ in range(48)],
    "task_id": [TASKS[i % 3] for i in range(48)],
})

FastLanguageModel.for_training(model)
cfg = GRPOConfig(
    output_dir=f"{OUTPUT_DIR}/grpo",
    num_generations=4, max_completion_length=1024,
    per_device_train_batch_size=2, gradient_accumulation_steps=4,
    num_train_epochs=1, learning_rate=5e-6, logging_steps=1, save_steps=50,
    bf16=torch.cuda.is_bf16_supported(), fp16=not torch.cuda.is_bf16_supported(),
    optim="adamw_8bit", warmup_steps=2, lr_scheduler_type="cosine",
    report_to="none",
)
trainer = GRPOTrainer(
    model=model, processing_class=tokenizer, train_dataset=ds,
    reward_funcs=REWARD_FUNCS, environment_factory=ClinicalRecruitmentToolEnv, args=cfg,
)
print("Starting GRPO refinement...")
trainer.train()
print("GRPO complete!")

## Final Evaluation + Save

In [ ]:
print(f"\n{sep}\nAFTER GRPO\n{sep}")
after_grpo = {}
for t in TASKS:
    r = run_episode(model, tokenizer, t, n=40)
    after_grpo[t] = r
    print(f"  [{t}] enrolled={r.get('enrolled',0)}/{r.get('target',0)} reward={r.get('total_reward',0)}")

# Comparison
CMP = ["total_reward", "enrolled"]
print(f"\n{sep}\nFULL COMPARISON\n{sep}")
for t in TASKS:
    b = {"total_reward": 0, "enrolled": 0}  # base model was zeros
    s, g = after_sft.get(t, {}), after_grpo.get(t, {})
    print(f"\n  [{t}]")
    print(f"    {'':>20}  {'Base':>10}  {'SFT':>10}  {'GRPO':>10}")
    for k in CMP:
        print(f"    {k:>20}  {b.get(k,0):>10.4f}  {s.get(k,0):>10.4f}  {g.get(k,0):>10.4f}")

# Save
lp = f"{OUTPUT_DIR}/lora_adapter"
model.save_pretrained(lp); tokenizer.save_pretrained(lp)
merged = model.merge_and_unload()
mp = f"{OUTPUT_DIR}/merged_model"
merged.save_pretrained(mp); tokenizer.save_pretrained(mp)
print(f"\nLoRA -> {lp} | Merged -> {mp}")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / "sft_grpo_results.json").write_text(json.dumps({
    "before": {"easy_bench": {"enrolled": 0}, "medium_bench": {"enrolled": 0}, "hard_bench": {"enrolled": 0}},
    "after_sft": after_sft, "after_grpo": after_grpo,
    "model": MODEL_NAME, "env_url": ENV_URL, "gpu": gpu}, indent=2))
print(f"\n{sep}\nDONE\n{sep}")